# 04 — Treinamento e Avaliação

Notebook atualizado para o modelo final de 3 classes: **decreto**, **lei** e **portaria**.

In [ ]:
import json
import torch
import torch.nn as nn
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay, accuracy_score

## 1. Carregar DataLoaders e modelo

In [ ]:
from src.dataset import carregar_datasets
from src.model import criar_modelo, contar_parametros

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
BATCH_SIZE = 8
EPOCAS = 20
LR = 1e-3

treino_loader, teste_loader, vocab, label_map, id2label = carregar_datasets(batch_size=BATCH_SIZE)
modelo = criar_modelo(vocab_size=len(vocab), num_classes=len(label_map)).to(DEVICE)

print('Dispositivo:', DEVICE)
print('Vocab:', len(vocab))
print('Classes:', label_map)
print('Parâmetros:', contar_parametros(modelo))

## 2. Configuração do treinamento

In [ ]:
criterio = nn.CrossEntropyLoss()
otimizador = torch.optim.Adam(modelo.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(otimizador, patience=5, factor=0.5)

## 3. Funções de treino e avaliação

In [ ]:
def treinar_epoca(modelo, loader):
    modelo.train()
    loss_total, acertos, total = 0, 0, 0
    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        otimizador.zero_grad()
        logits = modelo(x)
        loss = criterio(logits, y)
        loss.backward()
        otimizador.step()
        loss_total += loss.item() * y.size(0)
        acertos += (logits.argmax(1) == y).sum().item()
        total += y.size(0)
    return loss_total / total, acertos / total

def avaliar(modelo, loader):
    modelo.eval()
    loss_total, acertos, total = 0, 0, 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            logits = modelo(x)
            loss = criterio(logits, y)
            loss_total += loss.item() * y.size(0)
            acertos += (logits.argmax(1) == y).sum().item()
            total += y.size(0)
    return loss_total / total, acertos / total

## 4. Loop de treinamento

O resultado obtido no treinamento final do projeto foi **85,83%** de acurácia no conjunto de teste.

In [ ]:
historico = {'treino_loss': [], 'treino_acc': [], 'teste_loss': [], 'teste_acc': []}
melhor_acc = 0
Path('models').mkdir(exist_ok=True)

for epoca in range(1, EPOCAS + 1):
    tl, ta = treinar_epoca(modelo, treino_loader)
    vl, va = avaliar(modelo, teste_loader)
    scheduler.step(vl)
    
    historico['treino_loss'].append(tl)
    historico['treino_acc'].append(ta)
    historico['teste_loss'].append(vl)
    historico['teste_acc'].append(va)
    
    print(f'{epoca:02d} | treino_loss={tl:.4f} treino_acc={ta:.4f} teste_loss={vl:.4f} teste_acc={va:.4f}')
    
    if va >= melhor_acc:
        melhor_acc = va
        torch.save({
            'epoch': epoca,
            'model_state_dict': modelo.state_dict(),
            'optimizer_state_dict': otimizador.state_dict(),
            'vocab_size': len(vocab),
            'num_classes': len(label_map),
            'label_map': label_map,
            'id2label': id2label,
        }, 'models/modelo.pt')

print('Melhor acurácia:', melhor_acc)

## 5. Curvas de treinamento

In [ ]:
plt.plot(historico['treino_loss'], label='Treino loss')
plt.plot(historico['teste_loss'], label='Teste loss')
plt.title('Curva de Loss')
plt.xlabel('Época')
plt.ylabel('Loss')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

plt.plot(historico['treino_acc'], label='Treino acc')
plt.plot(historico['teste_acc'], label='Teste acc')
plt.title('Curva de Acurácia')
plt.xlabel('Época')
plt.ylabel('Acurácia')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## 6. Avaliação final

In [ ]:
def coletar_predicoes(modelo, loader):
    modelo.eval()
    y_true, y_pred = [], []
    with torch.no_grad():
        for x, y in loader:
            x = x.to(DEVICE)
            logits = modelo(x)
            y_pred.extend(logits.argmax(1).cpu().tolist())
            y_true.extend(y.tolist())
    return y_true, y_pred

y_true, y_pred = coletar_predicoes(modelo, teste_loader)
classes = [id2label[i] for i in range(len(id2label))]
print(classification_report(y_true, y_pred, target_names=classes))
print('Acurácia:', accuracy_score(y_true, y_pred))

In [ ]:
ConfusionMatrixDisplay.from_predictions(
    y_true, y_pred, display_labels=classes, cmap='Blues', xticks_rotation=45
)
plt.title('Matriz de Confusão')
plt.tight_layout()
plt.show()

## 7. Resultado final registrado

O treinamento final executado no projeto apresentou:

- Vocabulário: **17512 tokens**
- Classes: **decreto, lei e portaria**
- Treino/Teste: **480 / 120**
- Melhor acurácia: **85,83%**
- Parâmetros treináveis: **1.129.475**

| Classe | Precision | Recall | F1-score |
|--------|----------:|-------:|---------:|
| decreto | 0.89 | 0.78 | 0.83 |
| lei | 0.79 | 0.95 | 0.86 |
| portaria | 0.92 | 0.85 | 0.88 |